# Stage 0 — 평지 Goal 도달 학습 (로컬)

**담당**: 이재왕 (work/evader)  
**씬**: `Assets/01. Scenes/Stage0_Flat.unity`  
**목표**: 장애물·Pursuer 없는 평지에서 드론이 GoalZone에 도달하도록 학습  
**수렴 기준**: `goal_reach_rate ≥ 50%` → Stage0-B (Pursuer 추가)로 전환  

---

## 사전 준비 (최초 1회)

### 1. venv 활성화 후 Jupyter 실행
```bash
cd c:\IIT_DroneLearning
.venv\Scripts\activate
jupyter notebook python/notebooks/stage0_flat_local.ipynb
```
커널: **Python 3.10 (drone-evader)** 선택

### 2. Unity 씬 설정

| 항목 | 값 |
|---|---|
| 씬 경로 | `Assets/01. Scenes/Stage0_Flat.unity` |
| Behavior Name | `EvaderAgent` |
| Vector Obs Size | `18` |
| Continuous Actions | `4` |
| Decision Period | `5` |
| _goalOnlyMode | ✅ true |

### 3. 학습 흐름
1. **이 노트북 5번 셀 실행** → `mlagents-learn` 대기 상태 진입  
2. **Unity Editor에서 Play** → 자동 연결 후 학습 시작  
3. 6번 셀에서 TensorBoard 모니터링

---
## 1. 환경 확인

In [ ]:
import sys
import torch
import mlagents_envs

print(f'Python   : {sys.version.split()[0]}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'mlagents : {mlagents_envs.__version__}')

---
## 2. 경로 및 실험 설정

In [ ]:
import os
from pathlib import Path

# ── 로컬 경로 설정 ───────────────────────────────────────────────────────
REPO_PATH   = Path('c:/IIT_DroneLearning')
LOG_DIR     = REPO_PATH / 'python' / 'results'
CONFIG_DIR  = REPO_PATH / 'python' / 'config'

LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── 실험 설정 ────────────────────────────────────────────────────────────
SEED      = 42
INIT_FROM = None   # warm-start 필요 시 이전 run-id 문자열 입력

RUN_ID = f'evader_s0_flat_seed{SEED}'

print(f'Repo    : {REPO_PATH}')
print(f'Log dir : {LOG_DIR}')
print(f'Run ID  : {RUN_ID}')

---
## 3. Config 확인

In [ ]:
config_candidates = [
    CONFIG_DIR / 'evader_s0_flat_template.yaml',
    CONFIG_DIR / 'evader_s0_20260315_base.yaml',
    CONFIG_DIR / 'evader_s0_template.yaml',
]
config_path = next((c for c in config_candidates if c.exists()), None)
assert config_path, f'Config 없음: {config_candidates}'

print(f'Config: {config_path}\n{"=" * 60}')
print(config_path.read_text(encoding='utf-8'))

---
## 4. Unity Editor 연결 확인

> ⚠️ 다음 셀(5번) 실행 **전** Unity Editor에서 `Stage0_Flat.unity` 씬을 열어두세요.  
> 5번 셀 실행 → `mlagents-learn` 이 포트 5004 대기 → Unity에서 ▶ Play 누르면 자동 연결됩니다.

In [ ]:
import socket

port = 5004
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(('127.0.0.1', port))
    if result == 0:
        print(f'⚠️  포트 {port} 이미 사용 중 — 이전 mlagents-learn 프로세스가 남아있을 수 있습니다.')
    else:
        print(f'✅ 포트 {port} 사용 가능. 5번 셀 실행 후 Unity에서 Play 하세요.')

---
## 5. 학습 실행

> 이 셀을 실행하면 `mlagents-learn`이 Unity 연결을 대기합니다.  
> **Unity Editor → ▶ Play** 를 누르면 학습이 시작됩니다.

In [ ]:
import subprocess

cmd = [
    'mlagents-learn', str(config_path),
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
    '--force',
]

if INIT_FROM:
    cmd += [f'--initialize-from={INIT_FROM}']
    print(f'Warm-start from: {INIT_FROM}')

print('실행 커맨드:')
print(' '.join(cmd))
print()
print('Unity Editor에서 ▶ Play를 누르면 학습이 시작됩니다...')

result = subprocess.run(cmd, cwd=str(REPO_PATH))
print(f'\n학습 종료. Exit code: {result.returncode}')

---
## 6. TensorBoard 모니터링

학습 중 **새 터미널**에서 아래 명령어로 TensorBoard를 띄우거나, 아래 셀을 실행하세요.

In [ ]:
# 새 터미널에서 실행할 커맨드 출력
tb_cmd = f'.venv\\Scripts\\tensorboard --logdir python/results --port 6006'
print(f'터미널에서 실행:\n  {tb_cmd}')
print('\n브라우저에서 열기: http://localhost:6006')

# 노트북 내에서 바로 실행하려면 아래 주석 해제 (5번 셀과 병렬 실행 불가)
# %load_ext tensorboard
# %tensorboard --logdir {LOG_DIR}

# 확인 지표:
#   Environment/Cumulative Reward  → 상승 추세 (> 0.5 안정이면 수렴)
#   Environment/Episode Length     → 감소 추세 (더 빨리 Goal 도달)
#   Policy/Entropy                 → 초반 높고 서서히 감소
#
# 수렴 기준: Cumulative Reward > 0.5 안정
# → eval로 goal_reach_rate >= 50% 확인 후 Stage0-B 전환

---
## 7. 결과 확인 및 ONNX 저장 경로

In [ ]:
run_dir = LOG_DIR / RUN_ID

if run_dir.exists():
    onnx_files = list(run_dir.glob('**/*.onnx'))
    pt_files   = list(run_dir.glob('**/*.pt'))

    print(f'✅ 결과 폴더: {run_dir}')
    print(f'ONNX 파일 ({len(onnx_files)}개):')
    for f in onnx_files:
        print(f'  {f}')
    print(f'체크포인트 ({len(pt_files)}개):')
    for f in pt_files:
        print(f'  {f}')
else:
    print(f'❌ 결과 폴더 없음: {run_dir}')
    print('5번 셀(학습)을 먼저 실행하세요.')

---
## 8. Stage0-B 전환 (수렴 확인 후)

수렴 기준:
- TensorBoard `Cumulative Reward > 0.5` 안정
- `goal_reach_rate >= 50%`

**Unity Inspector 변경:**
- `EvaderAgent._goalOnlyMode` = **false** (체크 해제)
- `EvaderAgent._pursuerTransform` = ScriptedEvader 오브젝트 연결

**2번 셀(실험 설정) 변경:**
```python
INIT_FROM = 'evader_s0_flat_seed42'   # Stage0-A run-id
RUN_ID    = f'evader_s0b_withpursuer_seed{SEED}'
```

5번 셀 재실행 → Unity Play.

**실험 결과 기록:**
```bash
git add docs/EXPERIMENTS.md
git commit -m '[Docs] Stage0 평지 실험 결과 기록'
git push origin work/evader
```